In [18]:
import numpy as np
import pandas as pd
import math
import time
import os

%run utils/segmentation/detect.ipynb

weights = "yolov5s-seg.pt"
path = "utils/segmentation/"
os.chdir(path)
model = loadModel(weights)
path = "../.."
os.chdir(path)

Fusing layers... 
YOLOv5s-seg summary: 224 layers, 7611485 parameters, 0 gradients, 26.4 GFLOPs


In [58]:
# Class for an a list of objects 
class Object_List:
    def __init__(self):
        
        self.list = []
        self.archiveList = []
        
        self.rgb = None
        self.thermal = None
        self.stereo = None
        self.tol = 0.12
        self.thermalCap = 130
        
        self.csv = None
        
        # Depth Related Figures
        self.THRESH_LOW = 200 # 20cm
        self.THRESH_HIGH = 30000 # 30m
        
        ###### VALUE IS ONLY FOR OAK-D S2 | EXTRACTED FROM CALC.PY in stereoAI | Print value when with recording device ######
        self.monoHFOV = 1.2541936111357697
       
    
    # Calclates and angle of how far and deep an object is in an image
    def _calc_angle(self, frame, offset):
        return math.atan(math.tan(self.monoHFOV / 2.0) * offset / (frame.shape[1] / 2.0))
    
    
    # Gives the direction the object is moving. Used to determine if going off screen
    def _calc_derivative(self, obj):
        return obj.location_history[-1][0] - obj.location_history[-2][0]
    
    
    # Function to determine if two points are within a predetermined range with one another
    def inRange(self, p1, p2): # p1 newpoint p2 currentpoint
        yscale = 1
        
        xbool = p1[0] - self.tol <= p2[0] <= p1[0] + self.tol
        ybool = p1[1] - self.tol * yscale <= p2[1] <= p1[1] + self.tol * yscale
        
        return xbool and ybool
     
        
    # Counts how long an object as been "alive" for
    def addAlive(self):
        for obj in self.list:
            
            if len(obj.location_history) > 1:
                derv = self._calc_derivative(obj)
            
                # If the object is going offscreen achive obj
                if obj.location_history[-1][0] + derv >= 1 or obj.location_history[-1][0] + derv <= 0:
                    self.archiveList.append(obj)
                    self.list.remove(obj)

                    continue
                
            
            obj.framesAlive += 1
    
    
    # Goes through all detects to determine if it is an exisiting object
    def appendObjects(self, imgs, frame, detections, masks):
        self.addAlive()
        self.thermal = imgs[0]
        self.stereo = imgs[1]
        self.rgb = imgs[2]
        
        if detections == None or masks == None:
            return
        
        # Loop through all detections
        for det, mask in zip(detections, masks):
            
            # Grab the class, center, and mask of the object for later input
            cls, *center = det
            center = center[:2]
            mask = np.array(mask)
            
            # Assume a new object must be required
            newObj = True
            
            # Check all current objects in list
            for i, obj in enumerate(self.list):
                
                # If the object is not the same class or obj already has had frame inputed then ignore and moveon
                if cls != obj.cls or obj.frame_history[-1] == frame:
                    continue
                
                ### Rename this varible to something more discriptive
                # Current object centerial location
                c = obj.location_history[-1]
                
                ### Has to be a better way. Maybe checking size or ratio of object along side range
                # If objects are within range of one another assume same object
                if self.inRange(c, center):
                    
                    # Gather important infomation for object
                    properties = [obj.cls, frame, center, mask]
                    
                    # Update object with new data
                    self.list[i] = self.update(obj, *properties)
                    
                    # Matching object found. New object no longer required
                    newObj = False
                
                # If matched object found then exit function
                if not newObj:
                    break

            # If new object is found then create new object with None parameter
            if newObj:
                
                properties = [cls, frame, center, mask]
                
                # Create object then add it to list
                self.list.append( self.update(None, *properties) )
                            

    # Method to view the mask of a particular object in the list
    def viewMask(self, objNum):
        
        # Create copy of rgb image to mask over
        c = self.rgb.copy()
        
        # Grab the mask and resize it appropriately to image
        mask = self.list[objNum].mask
        mask = cv2.resize(mask, (c.shape[1], c.shape[0]))
        
        # Invert mask to delete unrequired pixels
        mask = 1-mask
        c[ mask.astype(bool) ] = 0
        
        # Display image after non-object pixels were deleted
        cv2.imshow("MaskedObj", c)
        cv2.waitKey(1)
    
    
    ### Later add method to determine if distance is valid or not. Zeros etc
    # Calculate how far an object in an image is
    def calc_spatials(self, depthFrame, mask, center, averaging_method=np.mean):
        cmTOm = 1000

        depthFrame = depthFrame / 255 * 30000
        
        # Calculate the average depth in the mask
        averageDepth = averaging_method(depthFrame[mask.astype(bool)])

        midX = int(depthFrame.shape[1] / 2) # middle of the depth img width
        midY = int(depthFrame.shape[0] / 2) # middle of the depth img height
        bb_x_pos = center[0] * depthFrame.shape[1] - midX
        bb_y_pos = center[1] * depthFrame.shape[0] - midY

        angle_x = self._calc_angle(depthFrame, bb_x_pos)
        angle_y = self._calc_angle(depthFrame, bb_y_pos)

        spatials = {
            'z': averageDepth/cmTOm,
            'x': averageDepth * math.tan(angle_x)/cmTOm,
            'y': -averageDepth * math.tan(angle_y)/cmTOm
        }
        
        return spatials
        
    def update(self, obj, cls, frame, center, mask):
        
        rgb = self.rgb.copy()
        thermal = self.thermal.copy()
        stereo = self.stereo.copy()
        
        m = mask.copy()
        m = cv2.resize(m, (rgb.shape[1], rgb.shape[0])).astype(bool)
        
        r = int( np.mean( rgb[m][:,0] ) )
        g = int( np.mean( rgb[m][:,1] ) ) 
        b = int( np.mean( rgb[m][:,2] ) )
        
        color = [r, g, b]
        
        temperature = np.mean( thermal[m] ) / 255 * self.thermalCap
        
        spatials = self.calc_spatials(stereo, m, center)
        
        idenity = len(self.list) + len(self.archiveList)
        
        properties = [cls, frame, center, color, temperature, spatials, mask, idenity]
        
        if obj == None:
            obj = Object(*properties)
            print(obj.id, spatials, center)
            
        else:
            obj.update(*properties)
            print(obj.id, spatials, center)
        
        return obj
        
    def createDataframe(self):
        
        master = []
        fullObjList = self.list + self.archiveList
        
        for i, obj in enumerate(fullObjList):
            location_history = np.array(obj.location_history)
            
            df = pd.DataFrame({'frame':obj.frame_history,
                               'objNum': obj.id,
                               'cls': obj.cls,
                               'x': obj.x,
                               'y': obj.y,
                               'z': obj.z,
                               'color': obj.color_history,
                               'thermal': obj.thermal_history})
            
            master.append(df)
        
        master = pd.concat(master)          
        
        master = master.sort_values(by = ['frame', 'objNum'])
        
        master = master.reset_index(drop=True)
        
        display(master)
        
        return master
            
class Object:
    def __init__(self, cls, frame, current_location, color, temperature, spatials, mask, idenity):
        
        self.cls = cls
        self.id = idenity
        
        self.mask = mask
        self.framesAlive = 0
        
        self.frame_history = [frame]
        self.location_history = [current_location]
        self.color_history = [color]
        self.thermal_history = [temperature]
        
        self.x = [spatials['x']]
        self.y = [spatials['y']]
        self.z = [spatials['z']]
                
    def update(self, cls, frame, current_location, color, temperature, spatials, mask, idenity):
        
        # print(cls, frame, [*current_location, depth], color, temperature, depth, mask.shape)
        self.mask = mask
        
        self.frame_history.append(frame)
        self.location_history.append(current_location)
        self.color_history.append(color)
        self.thermal_history.append(temperature)
        
        self.x.append(spatials['x'])
        self.y.append(spatials['y'])
        self.z.append(spatials['z'])
        
        

In [59]:
webVideo = cv2.VideoCapture("videos/2022-12-02_13-05-04/processed/rgb_sii.mp4")
thermalVideo = cv2.VideoCapture("videos/2022-12-02_13-05-04/processed/thermal_sii.mp4")
stereoVideo = cv2.VideoCapture("videos/2022-12-02_13-05-04/processed/stereo_sii.mp4")

ObjList = Object_List()

frame = 0
while(webVideo.isOpened()):

    ret, webframe = webVideo.read()
    ret1, thermalframe = thermalVideo.read()
    ret2, stereoframe = stereoVideo.read()
    
    if not ret and not ret1 and not ret2:
        break
    
    if frame == 0:
        camframe = webframe

    masks, det = detectObjects(webframe, model, classes=None, conf_thres=0.60)
    imgs = [thermalframe, stereoframe, webframe]
    
    ObjList.appendObjects(imgs, frame, det, masks)
    
    # ObjList.viewMask(0)

    cv2.imshow("s", webframe)
    
    prevWebFrame = webframe.copy()
    prevThermalFrame = thermalframe.copy()
    prevStereoFrame = stereoframe.copy()
    
    frame+=1
    # ig = input()
    if frame > 900:
        break
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

0 {'z': 10.885771754619501, 'x': -1.7872778012101287, 'y': -1.9875761754836778} [0.38671875, 0.6259765625]
1 {'z': 20.779858217436008, 'x': -11.146965010861464, 'y': -1.3823413074155375} [0.1298828125, 0.5458984375]
2 {'z': 26.788342385068464, 'x': 3.450341496325059, 'y': -0.3033267249516536} [0.5888671875, 0.5078125]
0 {'z': 10.475489798740277, 'x': -0.8006504388692698, 'y': -2.060933537089417} [0.447265625, 0.6357421875]
1 {'z': 22.53963846092227, 'x': -12.37808666758713, 'y': -1.5632119760612615} [0.12109375, 0.5478515625]
2 {'z': 26.40193518660728, 'x': 2.6905626244731344, 'y': -0.29895140271923715} [0.5703125, 0.5078125]
0 {'z': 10.009874022500355, 'x': 0.014167834000229368, 'y': -2.0968394320339465} [0.5009765625, 0.64453125]
1 {'z': 25.31296970932158, 'x': -14.080254274002723, 'y': -1.82720856990875} [0.1162109375, 0.5498046875]
2 {'z': 26.208419698377526, 'x': 2.1144164966549295, 'y': -0.3709502625710403} [0.5556640625, 0.509765625]
0 {'z': 9.880809725812586, 'x': 1.00693139195


KeyboardInterrupt



In [84]:
print(len(ObjList.archiveList))
print(len(ObjList.list))
# # print(ObjList.list[0].framesAlive)
x = ObjList.createDataframe()
x.to_csv("videos/2022-12-02_13-05-04/processed/processed.csv", index=False)

3
0


,frame,objNum,cls,x,y,z,color,thermal
0,0,0,2.0,-1.787278,-1.987576,10.885772,"[75, 76, 71]",90.308462
1,0,1,2.0,-11.146965,-1.382341,20.779858,"[100, 93, 84]",65.969347
2,0,2,2.0,3.450341,-0.303327,26.788342,"[90, 88, 79]",73.927093
3,1,0,2.0,-0.800650,-2.060934,10.475490,"[78, 80, 76]",89.085780
4,1,1,2.0,-12.378087,-1.563212,22.539638,"[103, 97, 87]",63.863490
...,...,...,...,...,...,...,...,...
77,44,2,2.0,3.426282,-2.351089,10.128622,"[107, 100, 92]",76.937003
78,45,2,2.0,4.088714,-2.340436,9.961251,"[110, 103, 94]",76.077672
79,46,2,2.0,4.207667,-1.902597,8.616828,"[103, 95, 88]",77.510157
80,47,2,2.0,5.331976,-2.112184,9.099405,"[101, 94, 90]",80.651553


In [82]:
webCopy = camframe.copy()

In [83]:
import time

s = camframe.shape[:2]
fourcc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')
trackOut = cv2.VideoWriter("videos/2022-12-02_13-05-04/processed/trackOut.mp4", fourcc, 10, (webCopy.shape[1], webCopy.shape[0]))

time.sleep(5)

for i, obj in enumerate(ObjList.archiveList):
    randcolor = [int(np.random.rand(1)[0] * 255), int(np.random.rand(1)[0] * 255), int(np.random.rand(1)[0] * 255)]
    
    for v in obj.location_history:
        p = v.copy()[:2]

        p[0] = int(s[1]*p[0])
        p[1] = int(s[0]*p[1])
        
        webCopy = cv2.circle(webCopy, p, 2, randcolor, 5)

        cv2.imshow("img", webCopy)
        cv2.waitKey(1)

        trackOut.write(webCopy)
    
        time.sleep(0.1)
    
trackOut.release()